# CombinedGenerator — XORing two MT19937s

`CombinedGenerator` XORs $J$ component generators bitwise to widen
the state space without paying the per-step cost of a single huge
generator. The combined characteristic polynomial is the product of
the components' polynomials (when pairwise coprime), so a
2-component combination of MT19937 with two distinct twist
parameters has state size $k_g = 2 \cdot 19937 = 39874$.

This notebook stamps two MT19937 instances with different twist
parameters $a$, combines them, and runs the equidistribution test on
the resulting 2-component combination.

See [theory/f2-linear-generators.md#combined-generators](../../theory/f2-linear-generators.md#combined-generators).


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest, METHOD_HARASE,
)
INT_MAX = 2**31 - 1

# Two MT19937 components with different `a` twist constants.
gen_a = Generator.create("MTGen", L=32,
    w=32, r=624, m=397, p=31, a=0x9908B0DF)  # canonical MT19937
gen_b = Generator.create("MTGen", L=32,
    w=32, r=624, m=397, p=31, a=0xDFFFFFFF)  # alternative twist

print(f"component A: k={gen_a.k}, L={gen_a.L}")
print(f"component B: k={gen_b.k}, L={gen_b.L}")

# 2-component Combination (J = 2).
comb = Combination(J=2, Lmax=32)
comb.components[0].add_gen(gen_a)
comb.components[1].add_gen(gen_b)
comb.reset()
print(f"combined: k_g = {comb.k_g}, L = {comb.L}")

# Equidistribution test on the combination — MT19937 has primitive
# χ_f, so the product is primitive too and Harase applies.
test = EquidistributionTest(
    L=comb.L,
    delta=[INT_MAX] * (comb.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)
print(f"SE (Σ gaps) = {result.se}")
print(f"verified    = {result.verified}")
